# 💥 Notebook 2: Crash and Recovery

In the last notebook we built a WAL but everything ran in one process. Here we *prove* it works by:

1. Creating a store, doing some writes.
2. Simulating a crash by **killing the process** without flushing.
3. Reopening the store and seeing what survived.

We compare two stores side by side:

- 🟥 **No-WAL store** — only flushes the whole dict periodically. Loses recent writes after a crash.
- 🟩 **WAL store** — fsyncs every operation to the log. Loses nothing acknowledged.

In [ ]:
import os, json, tempfile, subprocess, sys, textwrap
WORKDIR = tempfile.mkdtemp(prefix="wal_crash_")
print("workdir:", WORKDIR)

## ⚔️ The experiment

We launch a tiny Python child process that writes 100 keys and then **immediately exits without cleanup** (simulating a crash). Then we reopen the store and count surviving keys.

In [ ]:
script = textwrap.dedent('''
    import os, json, sys, signal

    workdir = sys.argv[1]
    mode = sys.argv[2]    # "wal" or "nowal"

    log_path = os.path.join(workdir, mode + ".log")
    snap_path = os.path.join(workdir, mode + ".snap")

    if mode == "nowal":
        # Snapshot mode: only periodically save the entire dict.
        data = {}
        for i in range(100):
            data[f"k{i}"] = i
            if i == 50:                  # only one snapshot, halfway
                with open(snap_path, "w") as f: json.dump(data, f); f.flush(); os.fsync(f.fileno())
        # crash WITHOUT a final snapshot:
        os.kill(os.getpid(), signal.SIGKILL)

    if mode == "wal":
        # WAL mode: fsync every write.
        log = open(log_path, "a")
        for i in range(100):
            log.write(json.dumps({"op": "put", "k": f"k{i}", "v": i}) + "\\n")
            log.flush(); os.fsync(log.fileno())
        os.kill(os.getpid(), signal.SIGKILL)
''')

script_path = os.path.join(WORKDIR, "child.py")
open(script_path, "w").write(script)

for mode in ("nowal", "wal"):
    print(f"--- running {mode} child ---")
    subprocess.run([sys.executable, script_path, WORKDIR, mode])
    # The child SIGKILL itself, so we'll see a non-zero exit. That's fine.

In [ ]:
# Recovery for the no-WAL store
nowal_path = os.path.join(WORKDIR, "nowal.snap")
nowal_data = json.load(open(nowal_path)) if os.path.exists(nowal_path) else {}
print(f"NO-WAL recovered {len(nowal_data)} keys (last snapshot was at i=50)")

# Recovery for the WAL store: replay the log
wal_path = os.path.join(WORKDIR, "wal.log")
wal_data = {}
if os.path.exists(wal_path):
    for line in open(wal_path):
        line = line.strip()
        if not line: continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            print("  ⚠️ skipping torn line"); continue
        if rec["op"] == "put": wal_data[rec["k"]] = rec["v"]
print(f"WAL    recovered {len(wal_data)} keys")

## ✅ What we just saw

- The **no-WAL** store lost everything written after the last snapshot. The more time between snapshots, the more data you lose.
- The **WAL** store recovered every acknowledged write. It pays the cost of one `fsync` per operation, but it never loses committed data.

This is *exactly* why every serious storage engine has a WAL.

In [ ]:
import shutil; shutil.rmtree(WORKDIR); print("cleaned up")